In [1]:
%pip install -q sentence-transformers scikit-learn scipy pandas numpy tqdm requests ir_datasets PyStemmer

In [2]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    raise RuntimeError("GPU is not enabled in Colab.")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [3]:
import os, re, sys, gc, json, math, time, random
import shutil, hashlib, tarfile, warnings, mmap
from array import array
from collections import defaultdict
import numpy as np
import pandas as pd
import scipy.sparse as sp
import requests
import torch
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# ---------- مسیرها (همان مسیر قبلی؛ collection.tsv و offsets.npy موجود دوباره استفاده می‌شوند) ----------
SAVE_DIR = "/content/msmarco_v1_full"
RAW_DIR = os.path.join(SAVE_DIR, "raw")
RUN_DIR = os.path.join(SAVE_DIR, "runs")
EMB_DIR = os.path.join(SAVE_DIR, "emb")
for d in (SAVE_DIR, RAW_DIR, RUN_DIR, EMB_DIR):
    os.makedirs(d, exist_ok=True)
# ---------- زیرمجموعهٔ کورپوس ----------
SUBSET_FRAC = 0.70      # کسری از ۸٫۸M پاساژ که هر دو سامانه روی آن اجرا می‌شوند (برای محدودیت زمانی Colab). 1.0 = کل کورپوس
SUB_MODE = "first"      # "first": ۷۰٪ اول پاساژها | "random": نمونهٔ تصادفی ۷۰٪
REUSE_OLD_BM25 = True   # اگر ایندکس BM25 کامل قبلی (پوشهٔ bm25_chunks) هست، chunkهای داخل ۷۰٪ اول دوباره ساخته نشوند (فقط حالت first)
OLD_IDX_DIR = os.path.join(SAVE_DIR, "bm25_chunks")
# ---------- ارزیابی ----------
N_DEV = 2000            # تعداد پرس‌وجوی نمونه‌گیری‌شده از dev.small
K_RET = 1000
SEED = 42
# ---------- BM25 ----------
BM25_K1 = 0.9
BM25_B = 0.4
CHUNK_DOCS = 50_000
BLOCK = 5_000
# ---------- Dense ----------
MODEL_NAME = "BAAI/bge-small-en-v1.5"
MAX_SEQ_LEN = 128
ENC_CHUNK = 100_000
ENC_BATCH = 512
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
assert torch.cuda.is_available()
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_num_threads(4)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
# ---------- ماژول تحلیل متن ----------
TEXT_MODULE = os.path.join(SAVE_DIR, "mw_text.py")
with open(TEXT_MODULE, "w") as f:
    f.write(r'''
import re
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, HashingVectorizer
_TOK = re.compile(r"\b\w+\b")
try:
    import Stemmer
    _STEM = Stemmer.Stemmer("english").stemWords
except Exception:
    _STEM = None
N_FEATURES = 2 ** 22
def analyzer(text):
    toks = [t for t in _TOK.findall(text.lower()) if t not in ENGLISH_STOP_WORDS]
    return _STEM(toks) if _STEM else toks
def make_vectorizer():
    return HashingVectorizer(n_features=N_FEATURES, alternate_sign=False, norm=None, analyzer=analyzer, dtype=np.float32)
def transform_block(texts):
    return make_vectorizer().transform(texts)
''')
sys.path.insert(0, SAVE_DIR)
import importlib, mw_text
importlib.reload(mw_text)
STEM_ON = mw_text._STEM is not None
print("Stemming:", STEM_ON, "| SAVE_DIR:", SAVE_DIR)

Stemming: True | SAVE_DIR: /content/msmarco_v1_full


In [4]:
ARCHIVE_URL = "https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz"
ARCHIVE = os.path.join(RAW_DIR, "collectionandqueries.tar.gz")
COLL_TSV = os.path.join(SAVE_DIR, "collection.tsv")
_ALT_TSV = os.path.join(RAW_DIR, "collection.tsv")   # مسیر جایگزین (نسخهٔ قبلی نوت‌بوک)
if not os.path.exists(COLL_TSV) and os.path.exists(_ALT_TSV):
    COLL_TSV = _ALT_TSV
OFF_NPY = os.path.join(SAVE_DIR, "offsets.npy")
def download(url, dest, chunk=1 << 20):
    if os.path.exists(dest) and os.path.getsize(dest) > 0.9 * 1024**3:
        return dest
    tmp = dest + ".part"
    done = os.path.getsize(tmp) if os.path.exists(tmp) else 0
    headers = {"Accept-Encoding": "identity"}
    if done:
        headers["Range"] = f"bytes={done}-"
    with requests.get(url, stream=True, headers=headers, timeout=(30, 120)) as r:
        if done and r.status_code != 206:
            done = 0
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0)) + done
        with open(tmp, "ab" if done else "wb") as f:
            with tqdm(total=total, initial=done, unit="B", unit_scale=True) as bar:
                for part in r.iter_content(chunk_size=chunk):
                    if part:
                        f.write(part)
                        bar.update(len(part))
    if os.path.getsize(tmp) <= 0.9 * 1024**3:
        raise RuntimeError("Archive download incomplete.")
    os.replace(tmp, dest)
    return dest
if not os.path.exists(COLL_TSV) and not os.path.exists(ARCHIVE):
    download(ARCHIVE_URL, ARCHIVE)
if not os.path.exists(COLL_TSV):
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        member = next((m for m in tar if os.path.basename(m.name) == "collection.tsv"), None)
        if member is None:
            raise FileNotFoundError("collection.tsv not found.")
        tmp = COLL_TSV + ".part"
        with tar.extractfile(member) as src, open(tmp, "wb") as dst:
            with tqdm(total=member.size, unit="B", unit_scale=True, desc="Extracting collection") as bar:
                while True:
                    buf = src.read(1 << 22)
                    if not buf:
                        break
                    dst.write(buf)
                    bar.update(len(buf))
        os.replace(tmp, COLL_TSV)
if os.path.exists(OFF_NPY):
    OFFSETS = np.load(OFF_NPY)
else:
    offs = array("q")
    pos = 0
    bad = 0
    with open(COLL_TSV, "rb") as f:
        for n, line in enumerate(tqdm(f, desc="Building offsets", total=8_841_823)):
            offs.append(pos)
            pos += len(line)
            tab = line.find(b"\t")
            if tab < 0 or int(line[:tab]) != n:
                bad += 1
    offs.append(pos)
    if bad:
        raise RuntimeError(f"Invalid passage IDs: {bad}")
    OFFSETS = np.frombuffer(offs, dtype=np.int64).copy()
    tmp = OFF_NPY + ".tmp"
    with open(tmp, "wb") as f:
        np.save(f, OFFSETS)
    os.replace(tmp, OFF_NPY)
N_ALL = len(OFFSETS) - 1
print("Total passages in MS MARCO v1:", f"{N_ALL:,}")
def read_pids(pids):
    """متن پاساژها را بر اساس شناسهٔ اصلی (شمارهٔ خط در collection.tsv) برمی‌گرداند."""
    out = []
    with open(COLL_TSV, "rb") as f:
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        try:
            for p in pids:
                s = int(OFFSETS[int(p)]); e = int(OFFSETS[int(p) + 1])
                line = mm[s:e]
                tab = line.find(b"\t")
                out.append(line[tab + 1:].rstrip(b"\r\n").decode("utf-8", "replace"))
        finally:
            mm.close()
    return out
print("Sample:", read_pids([0])[0][:150])

  0%|          | 0.00/1.06G [00:00<?, ?B/s]

Extracting collection:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Building offsets:   0%|          | 0/8841823 [00:00<?, ?it/s]

Total passages in MS MARCO v1: 8,841,823
Sample: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The onl


## گام ۳: بارگذاری پرس‌وجوها و qrels (DL19، DL20، dev.small)

In [5]:
import ir_datasets
SETS = {
    "DL19": ("msmarco-passage/trec-dl-2019/judged", 2),
    "DL20": ("msmarco-passage/trec-dl-2020/judged", 2),
    "dev.small": ("msmarco-passage/dev/small", 1),
}
ALL_Q, SET_SLICE, MIN_REL = [], {}, {}
rng = random.Random(SEED)
for name, (irid, min_rel) in SETS.items():
    ds = ir_datasets.load(irid)
    texts_q = {x.query_id: x.text for x in ds.queries_iter()}
    qrels = defaultdict(dict)
    for r in ds.qrels_iter():
        if r.relevance > 0:
            qrels[r.query_id][int(r.doc_id)] = r.relevance
    qids = [q for q in texts_q if any(v >= min_rel for v in qrels.get(q, {}).values())]
    if name == "dev.small" and N_DEV is not None and N_DEV < len(qids):
        qids = rng.sample(sorted(qids), N_DEV)
    start = len(ALL_Q)
    for q in qids:
        ALL_Q.append({"set": name, "qid": q, "text": texts_q[q], "rel": qrels[q]})
    SET_SLICE[name] = slice(start, len(ALL_Q))
    MIN_REL[name] = min_rel
    print(name, len(qids))
qtexts_all = [x["text"] for x in ALL_Q]
print("Total queries:", len(ALL_Q))

[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] [starting] https://trec.nist.gov/data/deep/2019qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2019qrels-pass.txt: [00:00] [187kB] [2.55MB/s]
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz: [00:00] [4.28kB] [25.0MB/s]


DL19 43


[INFO] [starting] https://trec.nist.gov/data/deep/2020qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2020qrels-pass.txt: [00:00] [219kB] [1.84MB/s]
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2020-queries.tsv.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2020-queries.tsv.gz: [00:00] [4.13kB] [21.6MB/s]
[INFO] If you have a local copy of https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/31644046b18952c1386cd4564ba2ae69
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz


DL20 54


[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: [00:29] [1.06GB] [35.3MB/s]


dev.small 2000
Total queries: 2097


## گام ۴: ساخت زیرمجموعهٔ ۷۰٪ کورپوس (یکسان برای BM25 و Dense)
حالت پیش‌فرض: ۷۰٪ اول پاساژها. پاساژهای مرتبط (qrels) پرس‌وجوهای ارزیابی که بیرون از آن می‌افتند هم اضافه می‌شوند (چند هزار پاساژ).

In [6]:
n_main = int(round(SUBSET_FRAC * N_ALL))
keep = np.zeros(N_ALL, dtype=bool)
if SUB_MODE == "first":
    keep[:n_main] = True
elif SUB_MODE == "random":
    keep[np.random.default_rng(SEED).permutation(N_ALL)[:n_main]] = True
else:
    raise ValueError("SUB_MODE must be 'first' or 'random'")
forced = np.fromiter({p for q in ALL_Q for p in q["rel"]}, dtype=np.int64)
assert forced.size and forced.min() >= 0 and forced.max() < N_ALL
n_added = int((~keep[forced]).sum())
keep[forced] = True
SUB_IDS = np.flatnonzero(keep).astype(np.int64)
NDOC = len(SUB_IDS)
SUB_HASH = hashlib.md5(SUB_IDS.tobytes()).hexdigest()[:10]
print(f"زیرمجموعه: {NDOC:,} از {N_ALL:,} پاساژ ({NDOC / N_ALL:.1%}) | حالت: {SUB_MODE} | پاساژ مرتبطِ اضافه‌شده: {n_added:,} | hash={SUB_HASH}")
# مسیرهای وابسته به زیرمجموعه (با تغییر SUBSET_FRAC خودکار جدا می‌شوند)
IDX_DIR = os.path.join(SAVE_DIR, f"bm25_sub_{SUB_HASH}")
EMB_ALL = os.path.join(EMB_DIR, f"emb_sub_{SUB_HASH}_fp16.npy")
EMB_META = os.path.join(EMB_DIR, f"emb_sub_{SUB_HASH}_meta.json")
COST_P = os.path.join(SAVE_DIR, f"cost_{SUB_HASH}.json")
os.makedirs(IDX_DIR, exist_ok=True)
def save_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)
def _cost_load():
    if os.path.exists(COST_P):
        with open(COST_P) as f:
            return json.load(f)
    return {}
def cost_add(key, val):
    c = _cost_load(); c[key] = float(c.get(key, 0.0)) + float(val); save_json(COST_P, c)
def cost_set(key, val):
    c = _cost_load(); c[key] = float(val); save_json(COST_P, c)
def to_orig(pos):
    """موقعیت در زیرمجموعه -> شناسهٔ اصلی پاساژ (برای ارزیابی با qrels)."""
    pos = np.asarray(pos)
    return np.where(pos >= 0, SUB_IDS[np.clip(pos, 0, None)], -1)

زیرمجموعه: 6,193,570 از 8,841,823 پاساژ (70.0%) | حالت: first | پاساژ مرتبطِ اضافه‌شده: 4,294 | hash=429c74a12e


## گام ۵: ساخت ایندکس BM25 روی زیرمجموعه (chunk به chunk، قابل ادامه)

In [7]:
META_P = os.path.join(IDX_DIR, "meta.json")
meta_now = {"stem": bool(STEM_ON), "chunk": int(CHUNK_DOCS), "block": int(BLOCK), "ndoc": int(NDOC),
            "nfeat": int(mw_text.N_FEATURES), "bm25_k1": float(BM25_K1), "bm25_b": float(BM25_B), "sub": SUB_HASH}
if os.path.exists(META_P):
    with open(META_P) as f:
        if json.load(f) != meta_now:
            raise RuntimeError("تنظیمات BM25 با ایندکس موجود فرق دارد؛ پوشهٔ IDX_DIR را پاک کنید.")
else:
    save_json(META_P, meta_now)
RANGES = [(s, min(s + CHUNK_DOCS, NDOC)) for s in range(0, NDOC, CHUNK_DOCS)]
print("Number of chunks:", len(RANGES))
def chunk_paths(ci):
    return os.path.join(IDX_DIR, f"tf_{ci:03d}.npz"), os.path.join(IDX_DIR, f"dl_{ci:03d}.npy")
# ---------- استفادهٔ مجدد از chunkهای ایندکس کامل قبلی (فقط chunkهایی که کاملاً داخل ۷۰٪ اول‌اند) ----------
reuse_ok = False
if REUSE_OLD_BM25 and SUB_MODE == "first" and os.path.exists(os.path.join(OLD_IDX_DIR, "meta.json")):
    with open(os.path.join(OLD_IDX_DIR, "meta.json")) as f:
        om = json.load(f)
    reuse_ok = (om.get("stem") == bool(STEM_ON) and om.get("chunk") == CHUNK_DOCS and om.get("nfeat") == mw_text.N_FEATURES)
    assert (SUB_IDS[:n_main] == np.arange(n_main)).all()
print("استفادهٔ مجدد از ایندکس قبلی:", reuse_ok)
n_reused = 0
for ci, (s, e) in enumerate(tqdm(RANGES, desc="Building BM25")):
    p_tf, p_dl = chunk_paths(ci)
    if os.path.exists(p_tf) and os.path.exists(p_dl):
        continue
    if reuse_ok and e <= n_main:
        o_tf, o_dl = (os.path.join(OLD_IDX_DIR, os.path.basename(p)) for p in (p_tf, p_dl))
        if os.path.exists(o_tf) and os.path.exists(o_dl) and np.load(o_dl, mmap_mode="r").shape[0] == e - s:
            for src_p, dst_p in ((o_dl, p_dl), (o_tf, p_tf)):
                if os.path.lexists(dst_p):
                    os.remove(dst_p)
                try:
                    os.symlink(src_p, dst_p)
                except OSError:
                    shutil.copy(src_p, dst_p)
            n_reused += 1
            continue
    for p in (p_tf, p_dl):
        if os.path.exists(p):
            os.remove(p)
    t_c = time.time()
    blocks = []
    for bs in range(s, e, BLOCK):
        be = min(bs + BLOCK, e)
        blocks.append(mw_text.transform_block(read_pids(SUB_IDS[bs:be])))
    X = sp.vstack(blocks, format="csr", dtype=np.float32)
    del blocks
    gc.collect()
    assert X.shape[0] == (e - s)
    dl = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    tmp_dl = p_dl + ".tmp"
    with open(tmp_dl, "wb") as f:
        np.save(f, dl)
    os.replace(tmp_dl, p_dl)
    tmp_tf = p_tf[:-4] + ".tmp.npz"
    sp.save_npz(tmp_tf, X, compressed=False)
    os.replace(tmp_tf, p_tf)
    del X, dl
    gc.collect()
    cost_add("bm25_index_sec", time.time() - t_c)
    cost_add("bm25_chunks_built", 1)
if n_reused:
    cost_set("bm25_chunks_reused", max(_cost_load().get("bm25_chunks_reused", 0), n_reused))
assert all(os.path.exists(p) for ci in range(len(RANGES)) for p in chunk_paths(ci)), "ایندکس BM25 ناقص است."
cost_set("bm25_index_gb", sum(os.path.getsize(p) for ci in range(len(RANGES)) for p in chunk_paths(ci)) / 1024**3)
print(f"BM25 index ready: {IDX_DIR} | chunkهای استفادهٔ مجدد: {n_reused} از {len(RANGES)}")

Number of chunks: 124
استفادهٔ مجدد از ایندکس قبلی: False


Building BM25:   0%|          | 0/124 [00:00<?, ?it/s]

BM25 index ready: /content/msmarco_v1_full/bm25_sub_429c74a12e | chunkهای استفادهٔ مجدد: 0 از 124


## گام ۶: آمار سراسری BM25 (df، طول اسناد، avgdl) روی زیرمجموعه

In [8]:
STATS_P = os.path.join(IDX_DIR, "stats.npz")
if os.path.exists(STATS_P):
    z = np.load(STATS_P)
    DF, DL = z["df"], z["dl"]
else:
    DF = np.zeros(mw_text.N_FEATURES, dtype=np.int32)
    DL_parts = []
    for ci in tqdm(range(len(RANGES)), desc="Global DF"):
        p_tf, p_dl = chunk_paths(ci)
        X = sp.load_npz(p_tf)
        DF += np.bincount(X.indices, minlength=mw_text.N_FEATURES).astype(np.int32)
        DL_parts.append(np.load(p_dl))
        del X
    DL = np.concatenate(DL_parts)
    np.savez(STATS_P + ".tmp.npz", df=DF, dl=DL)
    os.replace(STATS_P + ".tmp.npz", STATS_P)
    del DL_parts
    gc.collect()
assert len(DL) == NDOC
AVGDL = float(DL.mean())
IDF = np.log1p((NDOC - DF + 0.5) / (DF + 0.5)).astype(np.float32)
print("N:", NDOC, "| AVGDL:", round(AVGDL, 2), "| Unique features:", int((DF > 0).sum()))

Global DF:   0%|          | 0/124 [00:00<?, ?it/s]

N: 6193570 | AVGDL: 33.06 | Unique features: 989161


## گام ۷: جست‌وجوی BM25 روی کل زیرمجموعه (Top-1000)

In [9]:
def bm25_search(qtexts, K):
    Xq = mw_text.make_vectorizer().transform(qtexts).tocsr()
    terms = [Xq.indices[Xq.indptr[i]:Xq.indptr[i + 1]] for i in range(Xq.shape[0])]
    best_s = [np.empty(0, np.float32) for _ in qtexts]
    best_i = [np.empty(0, np.int64) for _ in qtexts]
    for ci, (s, e) in enumerate(tqdm(RANGES, desc="BM25 search")):
        X = sp.load_npz(chunk_paths(ci)[0]).tocsr()
        dl = DL[s:e]
        rows = np.repeat(np.arange(e - s, dtype=np.int32), np.diff(X.indptr))
        tf = X.data
        X.data = (IDF[X.indices] * tf * (BM25_K1 + 1) / (tf + BM25_K1 * (1 - BM25_B + BM25_B * dl[rows] / AVGDL))).astype(np.float32)
        del rows, tf
        W = X.tocsc()
        del X
        indptr, indices, data = W.indptr, W.indices, W.data
        acc = np.zeros(e - s, np.float32)
        for qi, tl in enumerate(terms):
            touched = []
            for t in tl:
                a, b = indptr[t], indptr[t + 1]
                if b > a:
                    idx = indices[a:b]
                    acc[idx] += data[a:b]
                    touched.append(idx)
            if not touched:
                continue
            cand = np.unique(np.concatenate(touched)) if len(touched) > 1 else touched[0]
            sc = acc[cand].copy()
            acc[cand] = 0
            if len(cand) > K:
                sel = np.argpartition(-sc, K - 1)[:K]
                cand, sc = cand[sel], sc[sel]
            ms = np.concatenate([best_s[qi], sc])
            mi = np.concatenate([best_i[qi], cand.astype(np.int64) + s])
            if len(ms) > K:
                sel = np.argpartition(-ms, K - 1)[:K]
                ms, mi = ms[sel], mi[sel]
            best_s[qi], best_i[qi] = ms, mi
        del W, indptr, indices, data, acc
        gc.collect()
    I = np.full((len(qtexts), K), -1, np.int64)
    S = np.zeros((len(qtexts), K), np.float32)
    for qi in range(len(qtexts)):
        o = np.lexsort((best_i[qi], -best_s[qi]))
        I[qi, :len(o)] = best_i[qi][o]
        S[qi, :len(o)] = best_s[qi][o]
    return I, S
run_fp = hashlib.md5(("|".join(qtexts_all) + f"|{BM25_K1}|{BM25_B}|{K_RET}|{STEM_ON}|{NDOC}|{SUB_HASH}").encode()).hexdigest()[:12]
BM25_RUN = os.path.join(RUN_DIR, f"bm25_{run_fp}.npz")
if os.path.exists(BM25_RUN):
    z = np.load(BM25_RUN)
    bm25_pos = z["I"]
    print("BM25 run از کش لود شد")
else:
    t0 = time.time()
    bm25_pos, bm25_S = bm25_search(qtexts_all, K_RET)
    cost_set("bm25_search_sec", time.time() - t0)
    np.savez(BM25_RUN + ".tmp.npz", I=bm25_pos, S=bm25_S)
    os.replace(BM25_RUN + ".tmp.npz", BM25_RUN)
    print("BM25 search time:", round(time.time() - t0), "seconds")
bm25_I = to_orig(bm25_pos)
print("BM25 shape:", bm25_I.shape)

BM25 search:   0%|          | 0/124 [00:00<?, ?it/s]

BM25 search time: 193 seconds
BM25 shape: (2097, 1000)


## گام ۸: معیارهای ارزیابی و نتایج BM25
معیارهای اصلی پروپوزال: NDCG@10 و MRR@10. معیارهای تکمیلی: nDCG@100، MAP، Recall@10/100/1000.
آستانهٔ مرتبط‌بودن برای معیارهای دودویی (MRR، MAP، Recall): DL19/DL20 سطح ≥۲ و dev.small سطح ≥۱.

In [10]:
DISC = 1.0 / np.log2(np.arange(2, 102))
def per_query_metrics(rankings, qs, min_rel):
    rows = []
    for ranking, q in zip(rankings, qs):
        rel = q["rel"]
        pos = {p for p, r in rel.items() if r >= min_rel}
        rk = [int(p) for p in ranking]
        ideal_all = sorted(rel.values(), reverse=True)
        def ndcg(k):
            gains = np.array([rel.get(p, 0) for p in rk[:k]], float)
            ideal = np.array(ideal_all[:k], float)
            idcg = (ideal * DISC[:len(ideal)]).sum()
            return (gains * DISC[:len(gains)]).sum() / idcg if idcg > 0 else 0.0
        first = next((i for i, p in enumerate(rk[:10]) if p in pos), None)
        hits, ap = 0, 0.0
        for i, p in enumerate(rk[:1000]):
            if p in pos:
                hits += 1
                ap += hits / (i + 1)
        row = {"MRR@10": 1.0 / (first + 1) if first is not None else 0.0,
               "nDCG@10": ndcg(10), "nDCG@100": ndcg(100),
               "MAP": ap / len(pos) if pos else 0.0}
        for k in (10, 100, 1000):
            row[f"R@{k}"] = sum(p in pos for p in rk[:k]) / len(pos) if pos else 0.0
        rows.append(row)
    return pd.DataFrame(rows)
def evaluate_all(run_I):
    return {name: per_query_metrics(run_I[sl], ALL_Q[sl], MIN_REL[name]) for name, sl in SET_SLICE.items()}
RUNS = {"BM25": bm25_I}
PER = {"BM25": evaluate_all(bm25_I)}
def show():
    rows = []
    for sysname, per in PER.items():
        for setn, df in per.items():
            rows.append({"set": setn, "system": sysname, "n": len(df), **df.mean().round(4).to_dict()})
    out = pd.DataFrame(rows).sort_values(["set", "system"]).reset_index(drop=True)
    display(out)
    return out
BM25_RESULTS = show()

,set,system,n,MRR@10,nDCG@10,nDCG@100,MAP,R@10,R@100,R@1000
0,DL19,BM25,43,0.6812,0.5169,0.5204,0.3077,0.1675,0.5252,0.7762
1,DL20,BM25,54,0.6629,0.5264,0.5437,0.3326,0.2823,0.6208,0.8275
2,dev.small,BM25,2000,0.2907,0.3253,0.3788,0.2971,0.4469,0.7047,0.8794


## گام ۹: انکد زیرمجموعهٔ ۷۰٪ با BGE-small روی GPU (fp16، قابل ادامه)

In [11]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(MODEL_NAME, device="cuda")
model.half()
model.max_seq_length = MAX_SEQ_LEN
DIM = model.get_sentence_embedding_dimension()
emb_now = {"model": MODEL_NAME, "max_len": MAX_SEQ_LEN, "ndoc": int(NDOC), "dim": int(DIM), "chunk": ENC_CHUNK, "sub": SUB_HASH}
done = set()
if os.path.exists(EMB_ALL) and os.path.exists(EMB_META):
    with open(EMB_META) as f:
        old = json.load(f)
    if {k: old.get(k) for k in emb_now} != emb_now:
        raise RuntimeError("تنظیمات انکدینگ عوض شده؛ فایل‌های emb_sub_* را پاک کنید.")
    done = set(old["done"])
    mm = np.lib.format.open_memmap(EMB_ALL, mode="r+")
else:
    mm = np.lib.format.open_memmap(EMB_ALL, mode="w+", dtype=np.float16, shape=(NDOC, DIM))
n_ch = math.ceil(NDOC / ENC_CHUNK)
for c in tqdm(range(n_ch), desc=f"Encoding {NDOC:,} passages (GPU)"):
    if c in done:
        continue
    t_c = time.time()
    s, e = c * ENC_CHUNK, min((c + 1) * ENC_CHUNK, NDOC)
    emb = model.encode(read_pids(SUB_IDS[s:e]), batch_size=ENC_BATCH, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=False)
    mm[s:e] = emb.astype(np.float16)
    mm.flush()
    done.add(c)
    save_json(EMB_META, {**emb_now, "done": sorted(done)})
    cost_add("dense_encode_sec", time.time() - t_c)
assert len(done) == n_ch
del mm
cost_set("dense_index_gb", os.path.getsize(EMB_ALL) / 1024**3)
print(f"انکدینگ کامل شد | حجم امبدینگ‌ها: {os.path.getsize(EMB_ALL) / 1024**3:.1f} GB")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 6,193,570 passages (GPU):   0%|          | 0/62 [00:00<?, ?it/s]

انکدینگ کامل شد | حجم امبدینگ‌ها: 4.4 GB


## گام ۱۰: بازیابی Dense مستقل روی کل زیرمجموعه (جست‌وجوی دقیق روی GPU)
شامل چک‌های صحت: سلامت فایل امبدینگ و تطبیق نتیجهٔ GPU با brute-force مستقل. هیچ ترکیبی با BM25 ساخته نمی‌شود.

In [12]:
VERIFY_DENSE = True   # چک brute-force روی ۲ پرس‌وجو (≈۱–۲ دقیقه)
with open(EMB_META) as f:
    emb_meta = json.load(f)
assert len(emb_meta["done"]) == math.ceil(NDOC / emb_meta["chunk"]), "انکدینگ ناقص است؛ گام ۹ را دوباره اجرا کنید."
E = np.load(EMB_ALL, mmap_mode="r")
assert E.shape == (NDOC, DIM) and E.dtype == np.float16, (E.shape, E.dtype)
probe = np.sort(np.random.default_rng(SEED).choice(NDOC, size=5000, replace=False))
norms = np.linalg.norm(np.asarray(E[probe], dtype=np.float32), axis=1)
assert norms.min() > 0.9 and norms.max() < 1.1, "ردیف صفر/نامعتبر در امبدینگ‌ها پیدا شد."
print(f"امبدینگ: {E.shape} {E.dtype} | norm: [{norms.min():.3f}, {norms.max():.3f}] | مدل: {emb_meta['model']}")
assert next(model.parameters()).dtype == torch.float16, "مدل باید fp16 باشد (مثل انکدینگ اسناد)."
t0 = time.time()
Q_EMB = model.encode([QUERY_PREFIX + t for t in qtexts_all], batch_size=128, normalize_embeddings=True,
                     convert_to_numpy=True).astype(np.float32)
cost_set("query_encode_sec", time.time() - t0)
assert Q_EMB.shape == (len(ALL_Q), DIM)
torch.backends.cuda.matmul.allow_tf32 = False   # fp32 دقیق
def dense_full_search(Q, K):
    q = torch.from_numpy(np.ascontiguousarray(Q)).to("cuda")
    nq = q.shape[0]
    step = max(20_000, int(5e8 // nq))
    best_s = torch.empty((nq, 0), dtype=torch.float32, device="cuda")
    best_i = torch.empty((nq, 0), dtype=torch.int64, device="cuda")
    for s in tqdm(range(0, NDOC, step), desc="Dense search (GPU)"):
        e = min(s + step, NDOC)
        ch = torch.from_numpy(np.array(E[s:e])).to("cuda").float()
        v, i = torch.topk(q @ ch.T, min(K, e - s), dim=1)
        best_s = torch.cat([best_s, v], 1)
        best_i = torch.cat([best_i, i + s], 1)
        if best_s.shape[1] > K:
            best_s, sel = torch.topk(best_s, K, dim=1)
            best_i = torch.gather(best_i, 1, sel)
        del ch, v, i
    torch.cuda.empty_cache()
    return best_i.cpu().numpy()
dense_fp = hashlib.md5((run_fp + json.dumps(emb_meta, sort_keys=True)).encode()).hexdigest()[:12]
DENSE_RUN = os.path.join(RUN_DIR, f"dense_{dense_fp}.npz")
if os.path.exists(DENSE_RUN):
    dense_pos = np.load(DENSE_RUN)["I"]
    print("Dense run از کش لود شد")
else:
    t0 = time.time()
    dense_pos = dense_full_search(Q_EMB, K_RET)
    cost_set("dense_search_sec", time.time() - t0)
    np.savez(DENSE_RUN + ".tmp.npz", I=dense_pos)
    os.replace(DENSE_RUN + ".tmp.npz", DENSE_RUN)
    print(f"Dense search time: {time.time() - t0:.0f} seconds")
assert dense_pos.shape == (len(ALL_Q), K_RET)
# ---------- چک‌های صحت ----------
def score(qi, ids):
    return np.asarray(E[np.sort(ids)], dtype=np.float32) @ Q_EMB[qi]
for qi in np.random.default_rng(SEED).choice(len(ALL_Q), size=min(100, len(ALL_Q)), replace=False):
    cand = bm25_pos[qi][bm25_pos[qi] >= 0]
    if len(cand) == 0:
        continue
    top1 = float(np.asarray(E[int(dense_pos[qi, 0])], dtype=np.float32) @ Q_EMB[qi])
    assert top1 >= score(qi, cand).max() - 1e-3, f"Dense top-1 ضعیف‌تر از یک کاندید BM25 است (q={qi})"
print("✔ Dense top-1 همیشه ≥ بهترین کاندید BM25 (چک سلامت جست‌وجوی دقیق)")
if VERIFY_DENSE:
    for qi in (0, len(ALL_Q) - 1):
        top, step = np.full(10, -9.0, np.float32), 500_000
        for s in range(0, NDOC, step):
            sims = np.asarray(E[s:min(s + step, NDOC)], dtype=np.float32) @ Q_EMB[qi]
            top = np.sort(np.concatenate([top, np.partition(sims, -10)[-10:]]))[-10:]
        mine = np.sort(np.asarray(E[np.sort(dense_pos[qi, :10])], dtype=np.float32) @ Q_EMB[qi])
        assert np.allclose(np.sort(top), mine, atol=2e-3), f"Dense مغایر با brute-force (q={qi})"
    print("✔ جست‌وجوی GPU با brute-force مستقل تطبیق داده شد")
dense_I = to_orig(dense_pos)

امبدینگ: (6193570, 384) float16 | norm: [1.000, 1.001] | مدل: BAAI/bge-small-en-v1.5


Dense search (GPU):   0%|          | 0/26 [00:00<?, ?it/s]

Dense search time: 20 seconds
✔ Dense top-1 همیشه ≥ بهترین کاندید BM25 (چک سلامت جست‌وجوی دقیق)
✔ جست‌وجوی GPU با brute-force مستقل تطبیق داده شد


## گام ۱۱: نتایج نهایی BM25 در برابر Dense + آزمون معناداری (paired bootstrap)

In [13]:
DENSE_NAME = "Dense (BGE-small)"
RUNS[DENSE_NAME] = dense_I
PER[DENSE_NAME] = evaluate_all(dense_I)
FINAL = show()
FINAL.to_csv(os.path.join(SAVE_DIR, "results_bm25_vs_dense.csv"), index=False)
print(f"\nپروتکل: encoder={MODEL_NAME} | fp16 | max_len={MAX_SEQ_LEN} | پاساژ={NDOC:,} ({NDOC / N_ALL:.0%} از {N_ALL:,}) | پرس‌وجو={len(ALL_Q)}")
def paired_bootstrap(a, b, n=10_000, seed=0, chunk=500):
    r = np.random.default_rng(seed)
    d = np.asarray(a) - np.asarray(b)
    m = np.concatenate([d[r.integers(0, len(d), size=(chunk, len(d)))].mean(axis=1) for _ in range(n // chunk)])
    lo, hi = np.percentile(m, [2.5, 97.5])
    return d.mean(), lo, hi, min(1.0, 2 * min((m <= 0).mean(), (m >= 0).mean()))
rows = []
for setn in SET_SLICE:
    for metric in ("nDCG@10", "MRR@10", "R@100"):
        a, b = PER[DENSE_NAME][setn][metric], PER["BM25"][setn][metric]
        d, lo, hi, p = paired_bootstrap(a, b)
        verdict = "Dense بهتر (معنادار)" if lo > 0 else ("BM25 بهتر (معنادار)" if hi < 0 else "تفاوت معنادار نیست")
        rows.append({"set": setn, "metric": metric, "BM25": round(b.mean(), 4), "Dense": round(a.mean(), 4),
                     "Δ (Dense−BM25)": round(d, 4), "CI95": f"[{lo:.4f}, {hi:.4f}]", "p": round(p, 4),
                     "Dense wins/ties/losses": f"{int((a > b + 1e-9).sum())}/{int((abs(a - b) <= 1e-9).sum())}/{int((a < b - 1e-9).sum())}",
                     "نتیجه": verdict})
SIG = pd.DataFrame(rows)
display(SIG)
SIG.to_csv(os.path.join(SAVE_DIR, "significance_bm25_vs_dense.csv"), index=False)

,set,system,n,MRR@10,nDCG@10,nDCG@100,MAP,R@10,R@100,R@1000
0,DL19,BM25,43,0.6812,0.5169,0.5204,0.3077,0.1675,0.5252,0.7762
1,DL19,Dense (BGE-small),43,0.8597,0.7197,0.6531,0.4597,0.2614,0.6320,0.8252
2,DL20,BM25,54,0.6629,0.5264,0.5437,0.3326,0.2823,0.6208,0.8275
3,DL20,Dense (BGE-small),54,0.8337,0.7121,0.6356,0.4812,0.3813,0.6950,0.8434
4,dev.small,BM25,2000,0.2907,0.3253,0.3788,0.2971,0.4469,0.7047,0.8794
5,dev.small,Dense (BGE-small),2000,0.4568,0.5065,0.5526,0.4602,0.6784,0.8937,0.9820



پروتکل: encoder=BAAI/bge-small-en-v1.5 | fp16 | max_len=128 | پاساژ=6,193,570 (70% از 8,841,823) | پرس‌وجو=2097


,set,metric,BM25,Dense,Δ (Dense−BM25),CI95,p,Dense wins/ties/losses,نتیجه
0,DL19,nDCG@10,0.5169,0.7197,0.2028,"[0.1100, 0.2972]",0.0000,34/0/9,Dense بهتر (معنادار)
1,DL19,MRR@10,0.6812,0.8597,0.1785,"[0.0397, 0.3203]",0.0114,16/22/5,Dense بهتر (معنادار)
2,DL19,R@100,0.5252,0.6320,0.1068,"[0.0268, 0.1821]",0.0094,26/3/14,Dense بهتر (معنادار)
3,DL20,nDCG@10,0.5264,0.7121,0.1857,"[0.1086, 0.2663]",0.0000,43/0/11,Dense بهتر (معنادار)
4,DL20,MRR@10,0.6629,0.8337,0.1708,"[0.0603, 0.2835]",0.0016,22/27/5,Dense بهتر (معنادار)
5,DL20,R@100,0.6208,0.6950,0.0742,"[0.0109, 0.1403]",0.0208,27/7/20,Dense بهتر (معنادار)
6,dev.small,nDCG@10,0.3253,0.5065,0.1812,"[0.1635, 0.1988]",0.0000,867/873/260,Dense بهتر (معنادار)
7,dev.small,MRR@10,0.2907,0.4568,0.1662,"[0.1479, 0.1843]",0.0000,855/883/262,Dense بهتر (معنادار)
8,dev.small,R@100,0.7047,0.8937,0.1890,"[0.1687, 0.2091]",0.0000,453/1484/63,Dense بهتر (معنادار)


## گام ۱۲: تحلیل هزینه و کارایی 
BM25 روی CPU اجرا می‌شود و Dense روی GPU؛ پس زمان‌ها را فقط به‌عنوان مقیاس هزینه بخوانید. زمان جست‌وجو برای کل پرس‌وجوها به‌صورت دسته‌ای اندازه‌گیری شده و ms/query میانگین سرشکن‌شده است.

In [14]:
C = _cost_load()
nq = len(ALL_Q)
g = lambda k: C.get(k, float("nan"))
built, reused = g("bm25_chunks_built"), C.get("bm25_chunks_reused", 0)
bm25_idx_min = (g("bm25_index_sec") / built * len(RANGES) / 60) if built and built > 0 else g("bm25_index_sec") / 60   # تخمین برای ساخت همهٔ chunkها
COST = pd.DataFrame([
    {"system": "BM25", "سخت‌افزار": "CPU", "ساخت ایندکس/انکدینگ (دقیقه)": bm25_idx_min,
     "حجم ایندکس/امبدینگ (GB)": g("bm25_index_gb"), "جست‌وجوی کل پرس‌وجوها (ثانیه)": g("bm25_search_sec"),
     "ms/query (سرشکن)": 1000 * g("bm25_search_sec") / nq},
    {"system": DENSE_NAME, "سخت‌افزار": "GPU " + torch.cuda.get_device_name(0), "ساخت ایندکس/انکدینگ (دقیقه)": g("dense_encode_sec") / 60,
     "حجم ایندکس/امبدینگ (GB)": g("dense_index_gb"), "جست‌وجوی کل پرس‌وجوها (ثانیه)": g("dense_search_sec") + g("query_encode_sec"),
     "ms/query (سرشکن)": 1000 * (g("dense_search_sec") + g("query_encode_sec")) / nq},
]).round(2)
display(COST)
COST.to_csv(os.path.join(SAVE_DIR, "cost_comparison.csv"), index=False)
if reused:
    print(f"توجه: {int(reused)} chunk از {len(RANGES)} chunk ایندکس BM25 از ایندکس کامل قبلی استفاده شد؛ زمان ساخت ایندکس BM25 بر اساس میانگین زمان chunkهای ساخته‌شده برای همهٔ chunkها تخمین زده شده است (برای زمان دقیق REUSE_OLD_BM25 = False بگذارید).")
print("توجه: اگر مرحله‌ای از کش لود شده یا با ادامهٔ اجرا انجام شده، زمان‌ها از فایل cost ثبت‌شده در همان اجراها خوانده می‌شوند.")

,system,سخت‌افزار,ساخت ایندکس/انکدینگ (دقیقه),حجم ایندکس/امبدینگ (GB),جست‌وجوی کل پرس‌وجوها (ثانیه),ms/query (سرشکن)
0,BM25,CPU,7.44,1.18,193.43,92.24
1,Dense (BGE-small),GPU Tesla T4,93.22,4.43,20.37,9.71


توجه: اگر مرحله‌ای از کش لود شده یا با ادامهٔ اجرا انجام شده، زمان‌ها از فایل cost ثبت‌شده در همان اجراها خوانده می‌شوند.


## گام ۱۳: تحلیل نوع پرس‌وجو و نمونه‌های موردی 
همپوشانی واژگانی = کسری از واژه‌های (stem‌شده، بدون stop-word) پرس‌وجو که در بهترین پاساژ مرتبط هم آمده‌اند. همپوشانی کم یعنی «شکاف واژگانی» بیشتر.

In [15]:
analyzer = mw_text.analyzer
rel_pids = sorted({p for q in ALL_Q for p, r in q["rel"].items() if r >= MIN_REL[q["set"]]})
rel_text = dict(zip(rel_pids, read_pids(rel_pids)))
pq = {name: pd.concat([PER[name][s] for s in SET_SLICE], ignore_index=True) for name in (("BM25", DENSE_NAME))}
rows = []
for qi, q in enumerate(ALL_Q):
    qt = set(analyzer(q["text"]))
    best = 0.0
    if qt:
        for p, r in q["rel"].items():
            if r >= MIN_REL[q["set"]]:
                best = max(best, len(qt & set(analyzer(rel_text[p]))) / len(qt))
    rows.append({"set": q["set"], "qid": q["qid"], "text": q["text"], "n_words": len(q["text"].split()), "overlap": best,
                 "b_ndcg": pq["BM25"].loc[qi, "nDCG@10"], "d_ndcg": pq[DENSE_NAME].loc[qi, "nDCG@10"],
                 "b_mrr": pq["BM25"].loc[qi, "MRR@10"], "d_mrr": pq[DENSE_NAME].loc[qi, "MRR@10"]})
QDF = pd.DataFrame(rows)
QDF["lexical_overlap_bin"] = pd.qcut(QDF["overlap"].rank(method="first"), 3, labels=["کم (شکاف واژگانی)", "متوسط", "زیاد"])
QDF["length_bin"] = pd.cut(QDF["n_words"], [0, 4, 7, 1000], labels=["کوتاه (≤4)", "متوسط (5–7)", "بلند (≥8)"])
def bin_table(col):
    gb = QDF.groupby(col, observed=True)
    t = gb.agg(n=("qid", "size"), BM25_nDCG10=("b_ndcg", "mean"), Dense_nDCG10=("d_ndcg", "mean"),
               BM25_MRR10=("b_mrr", "mean"), Dense_MRR10=("d_mrr", "mean"))
    t["Δ nDCG@10"] = t["Dense_nDCG10"] - t["BM25_nDCG10"]
    t["Dense wins"] = gb.apply(lambda x: int((x.d_ndcg > x.b_ndcg + 1e-9).sum()))
    t["BM25 wins"] = gb.apply(lambda x: int((x.b_ndcg > x.d_ndcg + 1e-9).sum()))
    return t.round(4)
print("بر اساس همپوشانی واژگانی پرس‌وجو و پاساژ مرتبط:")
BIN_OVERLAP = bin_table("lexical_overlap_bin"); display(BIN_OVERLAP)
print("بر اساس طول پرس‌وجو:")
BIN_LEN = bin_table("length_bin"); display(BIN_LEN)
QDF["delta"] = QDF["d_ndcg"] - QDF["b_ndcg"]
cols = ["set", "qid", "text", "overlap", "b_ndcg", "d_ndcg", "delta"]
print("۱۰ پرس‌وجویی که Dense بیشترین برتری را دارد:")
display(QDF.nlargest(10, "delta")[cols].round(3))
print("۱۰ پرس‌وجویی که BM25 بیشترین برتری را دارد:")
display(QDF.nsmallest(10, "delta")[cols].round(3))
BIN_OVERLAP.to_csv(os.path.join(SAVE_DIR, "analysis_by_overlap.csv"))
BIN_LEN.to_csv(os.path.join(SAVE_DIR, "analysis_by_length.csv"))

بر اساس همپوشانی واژگانی پرس‌وجو و پاساژ مرتبط:


,n,BM25_nDCG10,Dense_nDCG10,BM25_MRR10,Dense_MRR10,Δ nDCG@10,Dense wins,BM25 wins
lexical_overlap_bin,,,,,,,,
کم (شکاف واژگانی),699,0.1126,0.3656,0.0939,0.3161,0.2529,333,41
متوسط,699,0.3726,0.5604,0.3575,0.5337,0.1878,341,104
زیاد,699,0.5179,0.6225,0.4733,0.5746,0.1046,270,135


بر اساس طول پرس‌وجو:


,n,BM25_nDCG10,Dense_nDCG10,BM25_MRR10,Dense_MRR10,Δ nDCG@10,Dense wins,BM25 wins
length_bin,,,,,,,,
کوتاه (≤4),650,0.3595,0.5746,0.3311,0.5310,0.2151,312,81
متوسط (5–7),988,0.3311,0.5140,0.3076,0.4747,0.1829,452,132
بلند (≥8),459,0.3058,0.4380,0.2773,0.3955,0.1322,180,67


۱۰ پرس‌وجویی که Dense بیشترین برتری را دارد:


,set,qid,text,overlap,b_ndcg,d_ndcg,delta
115,dev.small,1049894,who sang i feel wonderful tonight,0.500,0.0,1.0,1.0
127,dev.small,29097,at what temperature should cooling fans start,0.750,0.0,1.0,1.0
135,dev.small,984178,where is hartwell ga,0.500,0.0,1.0,1.0
148,dev.small,1051339,what is medical aki,0.500,0.0,1.0,1.0
158,dev.small,1066043,why do we push others away,1.000,0.0,1.0,1.0
193,dev.small,716641,what is an open access university,0.667,0.0,1.0,1.0
197,dev.small,1085434,"what county is millville, pa in",0.667,0.0,1.0,1.0
199,dev.small,853057,what is the voltage in israel,0.500,0.0,1.0,1.0
202,dev.small,405867,is cerebral palsy a birth injury,1.000,0.0,1.0,1.0
214,dev.small,1091833,parceling definition,0.500,0.0,1.0,1.0


۱۰ پرس‌وجویی که BM25 بیشترین برتری را دارد:


,set,qid,text,overlap,b_ndcg,d_ndcg,delta
146,dev.small,1049329,what is nyu ranked,1.000,1.0,0.0,-1.0
173,dev.small,1091692,phone number for amtrak information,1.000,1.0,0.0,-1.0
254,dev.small,1083161,what does qhs stand for,0.667,1.0,0.0,-1.0
340,dev.small,538570,vvi airport code,0.667,1.0,0.0,-1.0
410,dev.small,141185,department of labor sedentary definition,0.750,1.0,0.0,-1.0
445,dev.small,1000519,where did the ottomans deport the armenians,0.750,1.0,0.0,-1.0
529,dev.small,155700,do clouds form in the vab,1.000,1.0,0.0,-1.0
567,dev.small,1094197,is cholecystitis fatal,1.000,1.0,0.0,-1.0
582,dev.small,342156,how thick is a 2x8 board,1.000,1.0,0.0,-1.0
797,dev.small,749244,what is frp lock on mean/,1.000,1.0,0.0,-1.0


## گام ۱۴: صحت‌سنجی معیارها با pytrec_eval 
اگر نصب pytrec_eval شکست بخورد، این سلول بی‌خطر رد می‌شود.

In [16]:
try:
    import subprocess
    try:
        import pytrec_eval
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytrec_eval-terrier"], check=True)
        import pytrec_eval
    rows = []
    for setn, sl in SET_SLICE.items():
        qs = ALL_Q[sl]
        qrel = {q["qid"]: {str(p): int(r) for p, r in q["rel"].items()} for q in qs}
        ev_n = pytrec_eval.RelevanceEvaluator(qrel, {"ndcg_cut_10"})
        ev_m = pytrec_eval.RelevanceEvaluator(qrel, {"recip_rank"}, relevance_level=MIN_REL[setn])
        for name, run_I in RUNS.items():
            part = run_I[sl]
            run = {q["qid"]: {str(int(p)): float(10 - i) for i, p in enumerate(part[k][:10]) if p >= 0} for k, q in enumerate(qs)}
            rn, rm = ev_n.evaluate(run), ev_m.evaluate(run)
            rows.append({"set": setn, "system": name,
                         "nDCG@10 (pytrec_eval)": round(np.mean([rn[q]["ndcg_cut_10"] for q in rn]), 4),
                         "nDCG@10 (کد من)": round(PER[name][setn]["nDCG@10"].mean(), 4),
                         "MRR@10 (pytrec_eval)": round(np.mean([rm[q]["recip_rank"] for q in rm]), 4),
                         "MRR@10 (کد من)": round(PER[name][setn]["MRR@10"].mean(), 4)})
    display(pd.DataFrame(rows))
except Exception as ex:
    print("چک pytrec_eval انجام نشد (مشکلی در نتایج اصلی نیست):", repr(ex)[:200])

,set,system,nDCG@10 (pytrec_eval),nDCG@10 (کد من),MRR@10 (pytrec_eval),MRR@10 (کد من)
0,DL19,BM25,0.5169,0.5169,0.6812,0.6812
1,DL19,Dense (BGE-small),0.7197,0.7197,0.8597,0.8597
2,DL20,BM25,0.5264,0.5264,0.6629,0.6629
3,DL20,Dense (BGE-small),0.7121,0.7121,0.8337,0.8337
4,dev.small,BM25,0.3253,0.3253,0.2907,0.2907
5,dev.small,Dense (BGE-small),0.5065,0.5065,0.4568,0.4568


In [ ]:

# نصب کتابخانه‌های موردنیاز
!pip install -q arabic-reshaper python-bidi

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import arabic_reshaper
from bidi.algorithm import get_display

# ------------------------------------------------
# 1. انتخاب فونت فارسی
# ------------------------------------------------
font_candidates = [
    "Vazirmatn",
    "Vazir",
    "Noto Sans Arabic",
    "Noto Naskh Arabic",
    "DejaVu Sans"
]

available_fonts = {f.name for f in fm.fontManager.ttflist}

font_name = next(
    (f for f in font_candidates if f in available_fonts),
    "DejaVu Sans"
)

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# ------------------------------------------------
# 2. تابع اصلاح نمایش فارسی
# ------------------------------------------------
def fa(text):
    reshaped = arabic_reshaper.reshape(str(text))
    return get_display(reshaped)

# ------------------------------------------------
# 3. داده‌های نمودار
# ------------------------------------------------
datasets = [
    "TREC DL 2019",
    "TREC DL 2020",
    "MS MARCO dev.small"
]

bm25 = [0.5169, 0.5264, 0.3253]
bge  = [0.7197, 0.7121, 0.5065]

x = np.arange(len(datasets))
width = 0.34

# ------------------------------------------------
# 4. رسم نمودار
# ------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5.5))

bars1 = ax.bar(
    x - width/2, bm25, width,
    label="BM25",
    color="#7895B2",
    edgecolor="#34495E",
    linewidth=0.8
)

bars2 = ax.bar(
    x + width/2, bge, width,
    label="BGE-small",
    color="#D98C72",
    edgecolor="#8C4A36",
    linewidth=0.8
)

# نمایش مقادیر روی ستون‌ها
ax.bar_label(bars1, fmt="%.4f", padding=4, fontsize=9)
ax.bar_label(bars2, fmt="%.4f", padding=4, fontsize=9)

# ------------------------------------------------
# 5. عنوان‌ها و برچسب‌های فارسی
# ------------------------------------------------
ax.set_title(
    fa("مقایسه عملکرد دو روش بازیابی بر اساس معیار nDCG@10"),
    fontsize=13,
    fontweight="bold",
    pad=18
)

ax.set_ylabel(
    fa("مقدار معیار nDCG@10"),
    fontsize=11,
    labelpad=10
)

ax.set_xlabel(
    fa("مجموعه‌های ارزیابی"),
    fontsize=11,
    labelpad=12
)

ax.set_xticks(x)
ax.set_xticklabels(datasets, fontsize=10)

# ------------------------------------------------
# 6. تنظیمات ظاهری
# ------------------------------------------------
ax.set_ylim(0, 0.85)
ax.set_yticks(np.arange(0, 0.81, 0.1))

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.35
)

ax.set_axisbelow(True)

ax.legend(
    loc="upper right",
    frameon=False,
    fontsize=10
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# ------------------------------------------------
# 7. ذخیره خروجی با کیفیت چاپ
# ------------------------------------------------
plt.savefig(
    "Figure_4_1_nDCG10_Comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Figure_4_1_nDCG10_Comparison.pdf",
    bbox_inches="tight"
)

plt.show()


In [ ]:

# ============================================================
# THESIS FIGURES — BM25 vs BGE-small
# Local macOS / Anaconda / Jupyter
# Output: PNG (300 DPI) + PDF
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pathlib import Path

# ------------------------------------------------------------
# 1. Local output directory
# ------------------------------------------------------------

OUTPUT_DIR = Path.cwd() / "Thesis_Figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

# ------------------------------------------------------------
# 2. Experimental results
# ------------------------------------------------------------

datasets = [
    "TREC DL 2019",
    "TREC DL 2020",
    "MS MARCO\ndev.small"
]

results = {
    "nDCG@10": {
        "BM25": [0.5169, 0.5264, 0.3253],
        "BGE-small": [0.7197, 0.7121, 0.5065]
    },

    "MRR@10": {
        "BM25": [0.6812, 0.6629, 0.2907],
        "BGE-small": [0.8597, 0.8337, 0.4568]
    },

    "Recall@100": {
        "BM25": [0.5252, 0.6208, 0.7047],
        "BGE-small": [0.6320, 0.6950, 0.8937]
    }
}

# ------------------------------------------------------------
# 3. Plot configuration
# ------------------------------------------------------------

available_fonts = {
    f.name for f in font_manager.fontManager.ttflist
}

if "Times New Roman" in available_fonts:
    selected_font = "Times New Roman"
elif "Liberation Serif" in available_fonts:
    selected_font = "Liberation Serif"
else:
    selected_font = "DejaVu Serif"

plt.rcParams.update({
    "font.family": selected_font,
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.linewidth": 0.9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False
})

COLORS = {
    "BM25": "#7F8C9A",
    "BGE-small": "#2878B5"
}

# ------------------------------------------------------------
# 4. Figure generation
# ------------------------------------------------------------

def create_comparison_figure(metric, values, filename):

    bm25 = np.array(values["BM25"])
    bge = np.array(values["BGE-small"])

    x = np.arange(len(datasets))
    width = 0.34

    fig, ax = plt.subplots(figsize=(9.2, 5.5))

    bars1 = ax.bar(
        x - width / 2,
        bm25,
        width,
        label="BM25",
        color=COLORS["BM25"],
        edgecolor="black",
        linewidth=0.65,
        zorder=3
    )

    bars2 = ax.bar(
        x + width / 2,
        bge,
        width,
        label="BGE-small",
        color=COLORS["BGE-small"],
        edgecolor="black",
        linewidth=0.65,
        zorder=3
    )

    # Numerical labels
    ax.bar_label(
        bars1,
        fmt="%.4f",
        padding=4,
        fontsize=9
    )

    ax.bar_label(
        bars2,
        fmt="%.4f",
        padding=4,
        fontsize=9
    )

    # Axes
    ax.set_ylabel(metric, labelpad=9)
    ax.set_xlabel("Evaluation Dataset", labelpad=10)

    ax.set_xticks(x)
    ax.set_xticklabels(datasets)

    ax.set_ylim(0, 1.0)
    ax.set_yticks(np.arange(0, 1.01, 0.1))

    # Grid
    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.6,
        alpha=0.35,
        zorder=0
    )

    # Legend
    ax.legend(
        loc="upper left",
        frameon=True,
        fancybox=False,
        edgecolor="0.75"
    )

    ax.margins(x=0.12)

    fig.tight_layout()

    # Save PNG
    png_path = OUTPUT_DIR / f"{filename}.png"

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    # Save PDF
    pdf_path = OUTPUT_DIR / f"{filename}.pdf"

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    print("PNG:", png_path)
    print("PDF:", pdf_path)


# ------------------------------------------------------------
# 5. Generate three figures
# ------------------------------------------------------------

create_comparison_figure(
    "nDCG@10",
    results["nDCG@10"],
    "Figure_4_1_nDCG10_Comparison"
)

create_comparison_figure(
    "MRR@10",
    results["MRR@10"],
    "Figure_4_2_MRR10_Comparison"
)

create_comparison_figure(
    "Recall@100",
    results["Recall@100"],
    "Figure_4_3_Recall100_Comparison"
)

# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print("\nAll figures generated successfully.")

for file in sorted(OUTPUT_DIR.iterdir()):
    print(f"{file.name} | {file.stat().st_size / 1024:.1f} KB")

# ============================================================
# FINAL THESIS FIGURES 4-4 & 5-4
# Soft pastel colors | Improved spacing | Publication quality
# macOS / Anaconda / Jupyter
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pathlib import Path

# ------------------------------------------------------------
# 1. Output directory
# ------------------------------------------------------------

OUTPUT_DIR = Path.cwd() / "Thesis_Figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Font configuration
# ------------------------------------------------------------

available_fonts = {
    f.name for f in font_manager.fontManager.ttflist
}

if "Times New Roman" in available_fonts:
    FONT = "Times New Roman"
elif "Liberation Serif" in available_fonts:
    FONT = "Liberation Serif"
else:
    FONT = "DejaVu Serif"

plt.rcParams.update({
    "font.family": FONT,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.labelcolor": "#454545",
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#B8B8B8",
    "xtick.color": "#555555",
    "ytick.color": "#555555",
})

# ------------------------------------------------------------
# 3. Soft pastel colors
# ------------------------------------------------------------

COLORS = {
    "BM25": "#B8C9D9",       # Soft pastel blue
    "BGE-small": "#B9D8CB"   # Soft pastel green
}

VALUE_COLOR = "#444444"
COUNT_COLOR = "#777777"
GRID_COLOR = "#E8EAED"

# ------------------------------------------------------------
# 4. Experimental results
# ------------------------------------------------------------

overlap_data = {
    "categories": [
        "Low overlap",
        "Medium overlap",
        "High overlap"
    ],
    "BM25": [0.1126, 0.3726, 0.5179],
    "BGE-small": [0.3656, 0.5604, 0.6225],
    "counts": [699, 699, 699]
}

length_data = {
    "categories": [
        "Short\n(<= 4 words)",
        "Medium\n(5-7 words)",
        "Long\n(>= 8 words)"
    ],
    "BM25": [0.3595, 0.3311, 0.3058],
    "BGE-small": [0.5746, 0.5140, 0.4380],
    "counts": [650, 988, 459]
}

# ------------------------------------------------------------
# 5. Figure generation function
# ------------------------------------------------------------

def create_analysis_figure(data, xlabel, filename):

    categories = data["categories"]
    bm25 = np.array(data["BM25"])
    bge = np.array(data["BGE-small"])

    x = np.arange(len(categories))
    width = 0.34

    fig, ax = plt.subplots(figsize=(10, 6.2))

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # BM25
    bars1 = ax.bar(
        x - width / 2,
        bm25,
        width,
        label="BM25",
        color=COLORS["BM25"],
        edgecolor="#9FB3C5",
        linewidth=0.8,
        zorder=3
    )

    # BGE-small
    bars2 = ax.bar(
        x + width / 2,
        bge,
        width,
        label="BGE-small",
        color=COLORS["BGE-small"],
        edgecolor="#9DBFAF",
        linewidth=0.8,
        zorder=3
    )

    # Value labels
    ax.bar_label(
        bars1,
        fmt="%.4f",
        padding=5,
        fontsize=9,
        color=VALUE_COLOR
    )

    ax.bar_label(
        bars2,
        fmt="%.4f",
        padding=5,
        fontsize=9,
        color=VALUE_COLOR
    )

    # Axes
    ax.set_ylabel("nDCG@10", labelpad=12)
    ax.set_xlabel(xlabel, labelpad=15)

    ax.set_xticks(x)
    ax.set_xticklabels(categories, linespacing=1.5)

    ax.set_ylim(0, 0.75)
    ax.set_yticks(np.arange(0, 0.76, 0.1))

    # Grid
    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.7,
        color=GRID_COLOR,
        zorder=0
    )

    ax.set_axisbelow(True)

    # Legend
    ax.legend(
        loc="upper left",
        frameon=True,
        facecolor="white",
        edgecolor="#E0E0E0",
        framealpha=1,
        fancybox=True,
        borderpad=0.8
    )

    # Sample counts: dedicated space below x-axis labels
    for i, count in enumerate(data["counts"]):

        ax.text(
            i,
            -0.205,
            f"n = {count}",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="center",
            fontsize=9,
            color=COUNT_COLOR,
            bbox=dict(
                boxstyle="round,pad=0.35",
                facecolor="#F5F6F7",
                edgecolor="#E5E7E9",
                linewidth=0.6
            ),
            clip_on=False
        )

    # Extra space for category labels and sample counts
    ax.set_xlim(-0.6, len(categories) - 0.4)

    fig.subplots_adjust(
        left=0.11,
        right=0.97,
        top=0.96,
        bottom=0.29
    )

    # Save PNG
    png_path = OUTPUT_DIR / f"{filename}.png"

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    # Save PDF
    pdf_path = OUTPUT_DIR / f"{filename}.pdf"

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    print("PNG:", png_path)
    print("PDF:", pdf_path)


# ------------------------------------------------------------
# 6. Figure 4-4: Lexical overlap
# ------------------------------------------------------------

create_analysis_figure(
    data=overlap_data,
    xlabel="Lexical Overlap Group",
    filename="Figure_4_4_Lexical_Overlap"
)

# ------------------------------------------------------------
# 7. Figure 5-4: Query length
# ------------------------------------------------------------

create_analysis_figure(
    data=length_data,
    xlabel="Query Length Group",
    filename="Figure_5_4_Query_Length"
)

# ------------------------------------------------------------
# 8. Completion
# ------------------------------------------------------------

print("\nBoth revised figures generated successfully.")



In [ ]:

# ============================================================
# THESIS FIGURES 6-4 & 7-4
# BM25 vs BGE-small
# Computational Cost and Retrieval Efficiency
# macOS / Anaconda / Jupyter
# Output: PNG (300 DPI) + PDF
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pathlib import Path

# ------------------------------------------------------------
# 1. Output directory
# ------------------------------------------------------------

OUTPUT_DIR = Path.cwd() / "Thesis_Figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Font configuration
# ------------------------------------------------------------

available_fonts = {
    f.name for f in font_manager.fontManager.ttflist
}

if "Times New Roman" in available_fonts:
    FONT = "Times New Roman"
elif "Liberation Serif" in available_fonts:
    FONT = "Liberation Serif"
else:
    FONT = "DejaVu Serif"

plt.rcParams.update({
    "font.family": FONT,
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.linewidth": 0.9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ------------------------------------------------------------
# 3. Attractive, soft colors
# ------------------------------------------------------------

COLORS = {
    "BM25": "#F2B880",       # Warm pastel orange
    "BGE-small": "#75BFC1"   # Fresh turquoise
}

EDGE_COLORS = {
    "BM25": "#D99A60",
    "BGE-small": "#539FA2"
}

TEXT_COLOR = "#404854"
GRID_COLOR = "#E8EBEF"

# ------------------------------------------------------------
# 4. Experimental results
# ------------------------------------------------------------

methods = ["BM25", "BGE-small"]

# Figure 6-4: Initial preparation time (minutes)
preparation_time = [7.44, 93.22]

# Figure 7-4: Average retrieval time per query (milliseconds)
average_query_time = [92.24, 9.71]

# ------------------------------------------------------------
# 5. General comparison plotting function
# ------------------------------------------------------------

def create_cost_figure(
    values,
    ylabel,
    filename,
    annotation,
    ylim=None
):

    x = np.arange(len(methods))

    fig, ax = plt.subplots(figsize=(8.5, 5.6))

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # Bars
    bars = ax.bar(
        x,
        values,
        width=0.52,
        color=[
            COLORS["BM25"],
            COLORS["BGE-small"]
        ],
        edgecolor=[
            EDGE_COLORS["BM25"],
            EDGE_COLORS["BGE-small"]
        ],
        linewidth=1.0,
        zorder=3
    )

    # Numerical labels
    for bar, value in zip(bars, values):

        ax.annotate(
            f"{value:.2f}",
            xy=(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height()
            ),
            xytext=(0, 9),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
            color=TEXT_COLOR
        )

    # Axis settings
    ax.set_ylabel(ylabel, labelpad=12)

    ax.set_xticks(x)
    ax.set_xticklabels(methods)

    if ylim is not None:
        ax.set_ylim(0, ylim)
    else:
        ax.set_ylim(0, max(values) * 1.22)

    # Grid
    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.7,
        color=GRID_COLOR,
        zorder=0
    )

    ax.set_axisbelow(True)

    # Annotation below chart
    ax.text(
        0.5,
        -0.17,
        annotation,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=9.5,
        color="#737B86"
    )

    # Layout
    fig.subplots_adjust(
        left=0.13,
        right=0.95,
        top=0.94,
        bottom=0.23
    )

    # Save PNG
    png_path = OUTPUT_DIR / f"{filename}.png"

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    # Save PDF
    pdf_path = OUTPUT_DIR / f"{filename}.pdf"

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    print("PNG saved:", png_path)
    print("PDF saved:", pdf_path)


# ------------------------------------------------------------
# 6. Figure 6-4: Initial preparation time
# ------------------------------------------------------------

create_cost_figure(
    values=preparation_time,
    ylabel="Preparation Time (minutes)",
    filename="Figure_6_4_Preparation_Time",
    annotation="Initial indexing / document embedding time",
    ylim=110
)

# ------------------------------------------------------------
# 7. Figure 7-4: Average retrieval time
# ------------------------------------------------------------

create_cost_figure(
    values=average_query_time,
    ylabel="Average Retrieval Time (ms/query)",
    filename="Figure_7_4_Average_Query_Time",
    annotation="Average search time per query",
    ylim=110
)

# ------------------------------------------------------------
# 8. Verification
# ------------------------------------------------------------

print("\nBoth figures generated successfully.")

for file in sorted(OUTPUT_DIR.glob("Figure_6_4*")):
    print(file.name)

for file in sorted(OUTPUT_DIR.glob("Figure_7_4*")):
    print(file.name)

